# 04 · Inference on new videos + latency

Runs a trained model on **any video file**: face clips (same code as training) → model → P(fake) → verdict at the
threshold chosen on validation in notebook 03. Every step is timed, on GPU and on CPU.

| Video source | What it does |
|---|---|
| `ffpp_sample` | downloads the first `N_PER_CLASS` videos per class with the official script (≈ 1 min). The `split` column shows whether a video was in training: only **test** videos are fair examples. |
| `upload` | upload your own files (e.g. a phone video of yourself) |
| `drive_folder` | all videos in `DRIVE_VIDEO_DIR` |

Requires notebook 03 for the run (threshold). Runtime: GPU (the CPU is measured as well).

In [ ]:
#@title 0 · Setup (run after every restart)
DRIVE_ROOT = "/content/drive/MyDrive/IS665_Deepfake"  #@param {type:"string"}
REPO_URL = "https://github.com/payelsarkarr/is665-team6-deepfake-recognition"  #@param {type:"string"}
from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess
os.makedirs(DRIVE_ROOT, exist_ok=True)
REPO = "/content/repo"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)       # newest code from GitHub
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO], check=True)
sys.path.insert(0, f"{REPO}/src")
import cv2
if not hasattr(cv2, "CascadeClassifier"):
    !pip -q install "opencv-python-headless<5"
    print("OpenCV 4 installed -> Runtime > Restart session, then run this cell again.")
META = "/content/FaceForensics/dataset"            # official split files
if not os.path.exists(META):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/ondyari/FaceForensics.git", "/content/FaceForensics"], check=True)
import importlib, ffpp
for m in ["data", "crops", "dataset", "models", "metrics", "train", "evaluate", "download", "infer"]:
    importlib.reload(importlib.import_module(f"ffpp.{m}"))
print("code:", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())

In [ ]:
#@title 1 · Settings
RUN = "mnv2_v1"  #@param {type:"string"}
WHICH = "best"  #@param ["best", "last"]
VIDEO_SOURCE = "ffpp_sample"  #@param ["ffpp_sample", "upload", "drive_folder"]
N_PER_CLASS = 3  #@param {type:"integer"}
DRIVE_VIDEO_DIR = f"{DRIVE_ROOT}/demo_videos"  #@param {type:"string"}
SCRIPT = f"{DRIVE_ROOT}/faceforensics_download_v4.py"
RUNS_DIR = f"{DRIVE_ROOT}/runs"
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (only CPU will be measured)")

In [ ]:
#@title 2 · Get the videos
from pathlib import Path
import pandas as pd
from ffpp.data import classify, load_official_meta
VID_DIR = Path("/content/infer_videos"); VID_DIR.mkdir(exist_ok=True)
if VIDEO_SOURCE == "ffpp_sample":
    from ffpp import download
    download.official(SCRIPT, str(VID_DIR / "ffpp"), num_videos=N_PER_CLASS)
elif VIDEO_SOURCE == "upload":
    from google.colab import files
    for name, data in files.upload().items():
        (VID_DIR / name).write_bytes(data)
else:
    VID_DIR = Path(DRIVE_VIDEO_DIR)
split_of, _, _ = load_official_meta(META)
rows = []
for p in sorted(q for q in VID_DIR.rglob("*") if q.suffix.lower() in (".mp4", ".mov", ".avi", ".mkv", ".webm")):
    method, target, _ = classify(p, VID_DIR)
    if method is None:                                  # not an FF++ file (e.g. your own upload)
        rows.append(dict(path=str(p), video=p.name, truth="unknown", method="-", split="-"))
    else:
        rows.append(dict(path=str(p), video=p.name, truth="real" if method == "original" else "fake",
                         method=method, split=split_of.get(target, "?")))
vids = pd.DataFrame(rows)
display(vids.drop(columns="path"))

In [ ]:
#@title 3 · Score every video
from ffpp.infer import Detector
det = Detector(f"{RUNS_DIR}/{RUN}", WHICH)
print(f"{RUN} ({det.cfg['model']}, epoch {det.epoch + 1}) | threshold {det.threshold:.3f} from {det.threshold_source} | {det.device}")
results = [det.score(p) for p in vids.path]
show = ["status", "score", "verdict", "n_clips", "duration_s", "t_extract_ms", "t_model_ms", "t_total_ms"]
table = pd.concat([vids.drop(columns="path"), pd.DataFrame(results)[show]], axis=1)
display(table.round(3))
out = Path(RUNS_DIR) / RUN / "eval"; out.mkdir(exist_ok=True)
table.to_csv(out / "inference_examples.csv", index=False)
print("saved", out / "inference_examples.csv")

In [ ]:
#@title 4 · Score timeline of one video (row number from the table above)
VIDEO_ROW = 0  #@param {type:"integer"}
import matplotlib.pyplot as plt
r = results[VIDEO_ROW]
assert r["status"] == "ok", r["status"]
n = r["n_clips"]
fig = plt.figure(figsize=(2.2 * n, 4.6))
for c in range(n):
    a = fig.add_subplot(2, n, c + 1); a.imshow(r["clip_faces"][c]); a.axis("off")
    a.set_title(f"clip {c + 1} @ {r['clip_start_s'][c]} s\nP(fake) {r['clip_scores'][c]:.2f}", fontsize=8)
a = fig.add_subplot(2, 1, 2)
if "frame_scores" in r:                                  # frame model: one score per frame
    for c, fs in enumerate(r["frame_scores"]):
        a.plot(range(c * len(fs), (c + 1) * len(fs)), fs, marker=".", lw=1.5, color="#2a78d6")
    a.set_xlabel("frame (16 per clip, ~25 fps)")
else:                                                    # clip model: one score per clip
    a.bar(range(1, n + 1), r["clip_scores"], color="#2a78d6"); a.set_xlabel("clip")
a.axhline(det.threshold, color="#e34948", ls="--", lw=1, label=f"threshold {det.threshold:.2f}")
a.set_ylim(0, 1); a.set_ylabel("P(fake)"); a.legend(frameon=False, fontsize=8)
a.spines[["top", "right"]].set_visible(False)
fig.suptitle(f"{r['video']}: {r['verdict']} (P(fake) = {r['score']:.2f}, truth: {vids.truth[VIDEO_ROW]})", fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
#@title 5 · Latency on GPU and CPU (median per video) -> runs/<RUN>/eval/latency.json
import json
rows = []
for dev in (["cuda"] if torch.cuda.is_available() else []) + ["cpu"]:
    det.to(dev).warmup()
    for p in vids.path:
        rows.append(det.score(p, timeline=False))
lat = pd.DataFrame(rows)
lat = lat[lat.status == "ok"]
cols = ["t_extract_ms", "t_prepare_ms", "t_model_ms", "t_total_ms", "duration_s"]
summary = lat.groupby("device")[cols].median().round(1)
summary["videos"] = lat.groupby("device").size()
display(summary)
print("t_extract = video decoding + face detection + cropping (always on the CPU);",
      "t_model = the model on the evaluation frames/clips only")
json.dump(dict(run=RUN, model=det.cfg["model"], cpu_threads=torch.get_num_threads(),
               gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
               median=summary.reset_index().to_dict("records")), open(out / "latency.json", "w"), indent=2)
lat.to_csv(out / "latency_per_video.csv", index=False)
print("saved", out / "latency.json")
det.to("cuda" if torch.cuda.is_available() else "cpu")